# Camera-aware shared depth: matched five-epoch pilot
**PARKING-SHARED-CAMERA-2026-10-27-r1**

Question: does giving the depth head the camera calibration improve 2–6 m distances? Both arms start at the same reviewed parking-balanced epoch 7. Both add the same 1,553-parameter residual adapter. The control receives a fixed reference camera; the experimental arm receives each image's calibration. The detector and projected corners remain frozen.

No teacher/KD, new dataset, held-out selection, scale fitting, export or phone update. Five epochs per arm, batch 8, fresh AdamW at 1e-4, Huber delta 0.2 m; same 80% near / 20% broad loss and sampler. Epoch 0 remains eligible for checkpoint selection. Results are development evidence, not qualification.

Use the existing CPU input ZIP and receipt plus the completed `parking_shared_near_results.zip` in Drive. Run sections 1–2 on CPU before allocating GPU. On reconnect, rerun from section 1: verified inputs/features are reused, and training resumes from the last completed epoch. A different GPU/package runtime requires a new RUN_ID, not edited receipts.

## 1. Mount Drive and update main (CPU)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
from pathlib import Path
import collections, datetime, hashlib, json, os, shlex, subprocess, sys
REVISION = 'PARKING-SHARED-CAMERA-2026-10-27-r1'
RUN_ID = 'shared_camera_5epoch_r1'
MOBILE = Path('/content/mobile_adas3d')
BASE = Path('/content/drive/MyDrive')
PACKAGE = BASE/'mobile_adas3d_inputs/parking_shared_near/trainval_inputs_r2'
ARCHIVE = PACKAGE/'shared_near_inputs.zip'
RECEIPT = PACKAGE/'archive_receipt.json'
WARMSTART = BASE/'mobile_adas3d_outputs/parking_shared_near/shared_depth_near_trainval_r1/parking_shared_near_results.zip'
INPUTS = Path('/content/shared_near_inputs_shared_depth_near_trainval_r1')
CACHE = Path('/content')/('shared_camera_features_'+RUN_ID)
REUSE_CACHE = Path('/content/shared_near_features_shared_depth_near_trainval_r1')
SOURCE = Path('/content/Urban_Shared_Near_Source')
WEIGHTS = Path('/content/urban_shared_near_weights')
OUTPUT = BASE/'mobile_adas3d_outputs/parking_shared_camera'/RUN_ID
PYTHON = sys.executable
SCRIPT = MOBILE/'scripts/train_parking_shared_camera.py'
DEPS = Path('/content')/('shared_camera_deps_'+RUN_ID)
OUTPUT.mkdir(parents=True, exist_ok=True)
def run(command, name, cwd=None):
    command = [str(x) for x in command]
    stamp = datetime.datetime.now(datetime.timezone.utc).strftime('%Y%m%dT%H%M%S_%fZ')
    log = OUTPUT/'colab_logs'/f'{name}_{stamp}.log'
    log.parent.mkdir(parents=True, exist_ok=True)
    print('+', shlex.join(command), '\nDurable log:', log, flush=True)
    env = os.environ.copy()
    env['PYTHONPATH'] = str(DEPS)+os.pathsep+str(MOBILE)
    env['YOLO_AUTOINSTALL'] = 'false'
    tail = collections.deque(maxlen=25)
    with log.open('w') as stream:
        p = subprocess.Popen(command, cwd=cwd, env=env, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
        try:
            for line in p.stdout:
                print(line, end='', flush=True); stream.write(line); stream.flush(); tail.append(line.rstrip())
        finally:
            p.stdout.close()
        code = p.wait(); stream.write(f'\nExit code: {code}\n')
    if code: raise RuntimeError(f'Exit {code}; preserve run and inspect {log}\n'+'\n'.join(tail))
def stage(name):
    run([PYTHON, '-X', 'faulthandler', '-u', SCRIPT, name, '--inputs', INPUTS, '--warmstart', WARMSTART, '--source', SOURCE, '--weights', WEIGHTS, '--cache', CACHE, '--reuse-cache', REUSE_CACHE, '--output', OUTPUT, '--device', 'cuda'], name, MOBILE)
url = '/'.join(['https:', '', 'github.com', 'Ali-RT', 'mobile_adas3d.git'])
if not MOBILE.exists(): run(['git','clone',url,MOBILE], 'clone')
branch = subprocess.check_output(['git','branch','--show-current'],cwd=MOBILE,text=True).strip()
if branch != 'main': raise RuntimeError(f'Use main, not {branch!r}; preserve other branches')
run(['git','pull','--ff-only'], 'update_main', MOBILE)
if not SCRIPT.is_file(): raise FileNotFoundError(SCRIPT)
print(REVISION, '\nOutput:', OUTPUT)


## 2. Verify and stage the existing package (CPU)
No raw nuScenes download or new split is needed. Four workers reuse verified local files. If the ZIP is missing, upload the existing ZIP + receipt to PACKAGE; do not activate GPU yet. If the warm-start ZIP is elsewhere, correct WARMSTART above.

In [ ]:
for path in [ARCHIVE, RECEIPT, WARMSTART]:
    if not path.is_file(): raise FileNotFoundError(f'CPU input missing: {path}. Do not activate GPU yet.')
def digest(path):
    h = hashlib.sha256()
    with Path(path).open('rb') as stream:
        while block := stream.read(1024*1024): h.update(block)
    return h.hexdigest()
receipt = json.loads(RECEIPT.read_text())
expected = 'ec3062ce299c7ac70e537e640b16950615bd0d08ed601c823decc28d2d87d1cb'
if receipt['sha256'] != expected or digest(ARCHIVE) != expected: raise RuntimeError('Input archive changed; preserve it')
run([PYTHON,MOBILE/'scripts/parking_shared_near_data.py','restore','--archive',ARCHIVE,'--output',INPUTS], 'restore_inputs', MOBILE)
stage('prepare')
print('CPU preparation passed. Now select a GPU runtime. Reconnection may clear /content; rerun sections 1–2 if needed.')


## 3. Pin task dependencies, run tests and native GPU smoke
Keep the runtime's Torch/CUDA; do not install old CUDA toolkits or Core ML. Only the reviewed Ultralytics and THOP are installed into a separate task directory. Actual CUDA execution is checked here before caching/training.

In [ ]:
run([PYTHON,'-m','pip','install','--quiet','--no-deps','--target',DEPS,'ultralytics==8.4.61','ultralytics-thop==2.0.18'], 'task_dependencies', MOBILE)
run([PYTHON,'-c',"import torch,cv2,numpy,PIL,scipy; from importlib.metadata import version; assert version('ultralytics')=='8.4.61'; assert torch.cuda.is_available(), 'Select a GPU runtime'; print(torch.__version__,torch.cuda.get_device_name(0)); x=torch.ones(8,device='cuda'); print(x.sum().item())"], 'cuda_preflight', MOBILE)
run([PYTHON,'-m','unittest','discover','-s','tests','-p','test_parking_shared_camera.py','-v'], 'camera_tests', MOBILE)
stage('smoke')
smoke = json.loads((OUTPUT/'runtime_smoke.json').read_text())
assert smoke['complete'] and smoke['both_arms_bit_exact_epoch7_parity'] and smoke['optimizer_steps']==0


## 4. Reuse or cache frozen image features once
The old cache is read-only and reused only when runtime, input and producer identities match and every sample verifies. Otherwise a separate local cache is created. Camera metadata is generated from calibration; no images or labels are recopied into feature files.

In [ ]:
stage('cache')


## 5. Train the two matched arms: five epochs each
1536 training and 256 development images, 192 optimizer updates per epoch, 960 per arm. Only depth decoder + adapter train. Checkpoints are saved after completed epochs; an interrupted epoch restarts from the previous checkpoint. The minimum development MAE selects a checkpoint, including epoch 0. No additional epochs are authorized by this notebook.

In [ ]:
stage('train')
result = json.loads((OUTPUT/'camera_pilot_results.json').read_text())
assert result['complete'] and result['frozen_detector_unchanged'] and not result['heldout_evaluated']
assert not result['deployment_qualified'] and not result['phone_app_changed']
print(json.dumps(result, indent=2))


## 6. Package and download results
Research checks: at least 5% development improvement over both initialization and the parameter-matched control; no cell worse by more than 5 cm; Vehicle 2–4 m beats the fixed-4 m diagnostic. These are experiment-review checks, not the product's ±10 cm acceptance threshold. Return the ZIP before any held-out evaluation, export or app change.

In [ ]:
stage('bundle')
BUNDLE = OUTPUT/'parking_shared_camera_results.zip'
print('Return this file:', BUNDLE)
from google.colab import files
files.download(str(BUNDLE))
